In this book, I process data which includes reading a video clip, getting skeletal sequence and storing those as numpy arrays. Also I perform data augumentation on entire dataset.

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.signal import medfilt
import pickle
import os
import cv2
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import random

In [ ]:
class Config():
    def __init__(self):        
        self.frame_l = 32 # the length of frames
        self.joint_n = 33 # the number of joints
        self.joint_d = 3 # the dimension of classes        
        self.data_dir = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3'
        self.save_dir = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save'
        self.aug_save_dir = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save/numpy_aug'
        self.data_info_file = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/dataset_info.txt'
        self.train_info_file = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save/aug_dataset_train.txt'
        self.test_info_file = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save/aug_dataset_test.txt'
C = Config()

In [ ]:
# Function to read dataset info from file
def read_dataset_info(file_path):
    dataset_info = []
    with open(file_path, "r") as info_file:
        for line in info_file:
            # Strip any leading or trailing whitespace and split the line by tabs
            info = line.strip().split("\t")
            dataset_info.append(info)
    return dataset_info

In [ ]:
# Define augmentation functions
def translate_skeleton(skeleton, max_translation=0.05):
    """Translate skeleton by a random amount."""
    translation = np.random.uniform(-max_translation, max_translation, size=(1, skeleton.shape[1], skeleton.shape[2]))
    return skeleton + translation

def scale_skeleton(skeleton, scale_range=(0.9, 1.1)):
    """Scale skeleton by a random factor."""
    scale_factor = np.random.uniform(*scale_range, size=(1, 1, skeleton.shape[2]))
    return skeleton * scale_factor

def rotate_skeleton(skeleton, max_angle=15):
    """Rotate skeleton by a random angle around the Z-axis."""
    angle = np.radians(np.random.uniform(-max_angle, max_angle))
    rotation_matrix = np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle), np.cos(angle), 0],
        [0, 0, 1]
    ])
    return np.dot(skeleton, rotation_matrix)

def apply_augmentations(skeleton, num_augments=5):
    augmented_skeletons = []

    if skeleton.size == 0:
        print("Warning: Empty skeleton array; no augmentations applied.")
        return augmented_skeletons

    #print("Original skeleton shape:", skeleton.shape)

    for _ in range(num_augments):
        augmented_skeleton = skeleton.copy()
        augmented_skeleton = translate_skeleton(augmented_skeleton)
        augmented_skeleton = scale_skeleton(augmented_skeleton)
        augmented_skeleton = rotate_skeleton(augmented_skeleton)
        augmented_skeletons.append(augmented_skeleton)

    return augmented_skeletons

In [ ]:
def extract_and_save_skeleton_data(input_folder, output_folder, data_info_file):
    import mediapipe as mp
    mp_holistic = mp.solutions.holistic
    os.makedirs(output_folder, exist_ok=True)  # Ensure the output folder exists

    with mp_holistic.Holistic(static_image_mode=False) as holistic, open(data_info_file, "w") as info_file:
        serial_number = 1
        for filename in os.listdir(input_folder):
            if filename.endswith(".mp4"):
                video_path = os.path.join(input_folder, filename)
                action_name = filename.split("_")[1]

                cap = cv2.VideoCapture(video_path)
                if not cap.isOpened():
                    print(f"Error: Failed to open video file {video_path}")
                    continue  # Skip this video and move to the next

                video_coords = []
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break

                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    results = holistic.process(rgb_frame)

                    if results.pose_landmarks:
                        frame_coords = [[landmark.x, landmark.y, landmark.z] for landmark in results.pose_landmarks.landmark]
                        video_coords.append(frame_coords)

                cap.release()

                if video_coords:
                    video_coords_array = np.array(video_coords)
                    original_filename = os.path.splitext(os.path.basename(video_path))[0]
                    original_npy_filename = f"{original_filename}_original.npy"
                    np.save(os.path.join(output_folder, original_npy_filename), video_coords_array)

                    info_file.write(f"{serial_number}\t{action_name}\t{original_npy_filename}\n")
                    serial_number += 1

if __name__ == "__main__":
    input_folder = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/sliced_clips'
    data_info_file = "dataset_info.txt"
    skeleton_data_folder = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save/numpy'

    # Extract skeleton data and save as .npy files
    extract_and_save_skeleton_data(input_folder, skeleton_data_folder, data_info_file)


In [ ]:
def draw_and_save_skeleton_images(input_folder, skeleton_data_folder, skeletal_image_folder):
    os.makedirs(skeletal_image_folder, exist_ok=True)  # Ensure the output folder exists

    for filename in os.listdir(input_folder):
        if filename.endswith(".mp4"):
            video_path = os.path.join(input_folder, filename)
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print(f"Error: Failed to open video file {video_path}")
                continue  # Skip this video and move to the next

            # Load corresponding skeleton data
            skeleton_filename = f"{os.path.splitext(filename)[0]}_original.npy"
            skeleton_filepath = os.path.join(skeleton_data_folder, skeleton_filename)
            if not os.path.exists(skeleton_filepath):
                print(f"Error: Skeleton data not found for {filename}")
                continue

            skeleton_data = np.load(skeleton_filepath)  # Load the saved skeleton data

            # Create a subfolder for the skeletal images
            video_name = os.path.splitext(filename)[0]
            video_skeletal_folder = os.path.join(skeletal_image_folder, video_name)
            os.makedirs(video_skeletal_folder, exist_ok=True)

            frame_count = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_count < len(skeleton_data):
                    frame_coords = skeleton_data[frame_count]  # Get joint coordinates for this frame
                    for x, y, _ in frame_coords:
                        x = int(x * frame.shape[1])
                        y = int(y * frame.shape[0])
                        cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

                    # Optionally connect landmarks based on your specific skeleton structure
                    connections = [
                                    # Head and Face
                                    (0, 1), (1, 2), (2, 3), (3, 7),  # Left eye
                                    (0, 4), (4, 5), (5, 6), (6, 8),  # Right eye
                                    (9, 10),  # Mouth
                                
                                    # Upper Body
                                    (11, 12),  # Shoulders
                                    (11, 13), (13, 15), (15, 17),  # Left arm
                                    (15, 19), (15, 21),  # Left fingers (pinky and index)
                                    (12, 14), (14, 16), (16, 18),  # Right arm
                                    (16, 20), (16, 22),  # Right fingers (pinky and index)
                                
                                    # Torso
                                    (11, 23), (12, 24),  # Connections from shoulders to hips
                                    (23, 24),  # Hips
                                
                                    # Lower Body
                                    (23, 25), (25, 27), (27, 29), (29, 31),  # Left leg
                                    (24, 26), (26, 28), (28, 30), (30, 32)   # Right leg
                                ]

                    for start_idx, end_idx in connections:
                        if start_idx < len(frame_coords) and end_idx < len(frame_coords):
                            start = frame_coords[start_idx]
                            end = frame_coords[end_idx]
                            start_point = (int(start[0] * frame.shape[1]), int(start[1] * frame.shape[0]))
                            end_point = (int(end[0] * frame.shape[1]), int(end[1] * frame.shape[0]))
                            cv2.line(frame, start_point, end_point, (0, 255, 0), 2)

                    skeletal_image_filename = f"frame_{frame_count}.png"
                    skeletal_image_path = os.path.join(video_skeletal_folder, skeletal_image_filename)
                    cv2.imwrite(skeletal_image_path, frame)

                frame_count += 1

            cap.release()

if __name__ == "__main__":
    input_folder = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/sliced_clips'
    skeleton_data_folder = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save/numpy'
    skeletal_image_folder = r'/home/nyam/Thesis_local/Autism/Dara_proc_v2/Trail3/Data_save/skeletal_images'


    # Draw skeletons using saved skeleton data and save annotated images into subfolders
    draw_and_save_skeleton_images(input_folder, skeleton_data_folder, skeletal_image_folder)

In [ ]:
all_video_data = read_dataset_info(C.data_info_file)

labels = [all_video_data[i][1] for i in range(len(all_video_data))]
# Split the collected data into training and testing sets
train_data, test_data = train_test_split(all_video_data, test_size=0.2, train_size = 0.8, random_state=42, stratify=labels)

# Save the original data for the test set (without augmentations)
with open(C.test_info_file, "w") as info_file: 
    for i,(serial_number, action_name, skel) in enumerate(test_data):
        # Write info about the original data to the file
        info_file.write(f"{serial_number}\t{action_name}\t{str(skel)}\n")
        
# Save the original data for the train set (without augmentations)
with open(C.train_info_file, "w") as info_file: 
    for i,(serial_number, action_name, skel) in enumerate(train_data):
        # Write info about the original data to the file
        info_file.write(f"{serial_number}\t{action_name}\t{str(skel)}\n")
        
print(len(test_data))
print(len(train_data))


In [ ]:
# Function to process multiple video clips
def Augu_video_clips(data_info_file,train_info_file, test_info_file):
        
    # Apply augmentations only to the training set
    for i, (serial_number, action_name, skel) in enumerate(train_data):
        skeleton_path = C.save_dir + '\\' + 'numpy' + '\\'+str(skel)
        try:
            skeleton = np.load(skeleton_path).astype('float32')
            np.save(os.path.join(C.aug_save_dir, str(skel)), skeleton)
        except Exception as e:
            print(f"An error occurred while processing {skeletons}: {e}")
        
        if action_name == 'HandAction' or action_name == 'NoAction':
            augmented_skeletons = apply_augmentations(skeleton, 1)
        # elif action_name == 'Spinning':
        #     augmented_skeletons = apply_augmentations(skeleton, 1)
        # else:
        #     augmented_skeletons = apply_augmentations(skeleton, 1)
            for j, augmented_skeleton in enumerate(augmented_skeletons):
                aug_npy_filename = f"{action_name}_aug_{i}_{j}.npy"  # Adjust filename for uniqueness
                np.save(os.path.join(C.aug_save_dir, aug_npy_filename), augmented_skeleton)
                # Write info about the augmented data to the file
                with open(train_info_file, "a") as info_file:  # Append to the existing info file
                    info_file.write(f"{serial_number}\t{action_name}\t{aug_npy_filename}\n")

if __name__ == "__main__":      
    #Augument data after splitting into train and test
    #Augu_video_clips(C.data_info_file, C.train_info_file, C.test_info_file)


In [ ]:
labels_train = read_dataset_info(C.train_info_file)
labels_test = read_dataset_info(C.test_info_file)

labels_train = [labels_train[i][1] for i in range(len(labels_train))]
labels_test = [labels_test[i][1] for i in range(len(labels_test))]

# Create subplots side by side
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot histogram for the training labels
axs[0].hist(sorted(labels_train), bins=len(set(labels_train)), edgecolor='black')
axs[0].set_title('Training Labels Distribution')
axs[0].set_xlabel('Label')
axs[0].set_ylabel('Frequency')

# Plot histogram for the testing labels
axs[1].hist(sorted(labels_test), bins=len(set(labels_test)), edgecolor='black')
axs[1].set_title('Testing Labels Distribution')
axs[1].set_xlabel('Label')
axs[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Function to perform label encoding
def label_encode(dataset_info):
    label_encoder = LabelEncoder()
    # Extract labels from the dataset info
    labels = [info[1] for info in dataset_info]
    # Fit label encoder and transform labels
    # print(labels)
    encoded_labels = label_encoder.fit_transform(labels)
                            
    for i, info in enumerate(dataset_info):
        info[1] = encoded_labels[i]
    return dataset_info


# Load dataset info from the output_info_file
train_dataset_info = label_encode(read_dataset_info(C.train_info_file))
test_dataset_info = label_encode(read_dataset_info(C.test_info_file))

# Save training dataset info to a text file
with open(C.train_info_file, "w") as train_file:
    for info in train_dataset_info:
        train_file.write("\t".join(map(str, info)) + "\n")

# Save test dataset info to a text file
with open(C.test_info_file, "w") as test_file:
    for info in test_dataset_info:
        test_file.write("\t".join(map(str, info)) + "\n")


In [ ]:
# Read training dataset info
train_list = read_dataset_info(C.train_info_file)

# Read test dataset info
test_list = read_dataset_info(C.test_info_file)

In [ ]:
Train = {}
Train['pose'] = []
Train['label'] = []

for i in tqdm(range(len(train_list))):
    label = train_list[i][1]
    skeletons = train_list[i][2]
    
    skeleton_path = C.save_dir + '/' + "numpy" + '/'+str(skeletons)
    
    try:
        p = np.load(skeleton_path).astype('float32')
        for j in range(p.shape[1]):
            p[:, j] = medfilt(p[:, j])
        
        Train['pose'].append(p)
        Train['label'].append(label)
    except Exception as e:
        print(f"An error occurred while processing {skeletons}: {e}")

# Save the same structured data as a pickle file
with open(C.save_dir + "/train.pkl", "wb") as f:
    pickle.dump(Train, f)
                
# Prepare data for saving in Excel and pickle formats
rows = []
for i in tqdm(range(len(Train['pose']))):
    label = Train['label'][i]
    video_name = f"video_{i}"  # Use video index as name; adjust if you have actual names
    pose_data = Train['pose'][i]

    # Iterate over each frame
    for frame_index in range(pose_data.shape[0]):
        row = {
            'video_name': video_name,
            'frame_number': frame_index
        }

        # Add each joint's (x, y, z) coordinate as a tuple in a separate column
        for joint_index in range(pose_data.shape[1]):
            row[f'joint_{joint_index + 1}'] = tuple(pose_data[frame_index, joint_index, :])
        
        row['label'] = label
        rows.append(row)

# Convert structured data into DataFrame for Excel
df_train = pd.DataFrame(rows)
df_train.to_excel(C.save_dir + "/train_aug_formatted.xlsx", index=False)

print("Data saved as both 'train_aug_formatted.xlsx' and 'train_aug_formatted.pkl' in the specified format.")


In [ ]:
len(Train['pose'])

In [ ]:
Test = {}
Test['pose'] = []
Test['label'] = []

for i in tqdm(range(len(test_list))):
    label = test_list[i][1]
    skeletons = test_list[i][2]
    
    skeleton_path = C.save_dir + '/' + 'numpy' + '/' + str(skeletons)
    
    try:
        p = np.load(skeleton_path).astype('float32')
        for j in range(p.shape[1]):
            p[:, j] = medfilt(p[:, j])
        
        Test['pose'].append(p)
        Test['label'].append(label)
    except Exception as e:
        print(f"An error occurred while processing {skeletons}: {e}")

# Save the same structured data as a pickle file
with open(C.save_dir + "/test.pkl", "wb") as f:
    pickle.dump(Test, f)

# Prepare data for saving in Excel and pickle formats
rows = []
for i in tqdm(range(len(Test['pose']))):
    label = Test['label'][i]
    video_name = f"video_{i}"  # Use video index as name; adjust if you have actual names
    pose_data = Test['pose'][i]

    # Iterate over each frame
    for frame_index in range(pose_data.shape[0]):
        row = {
            'video_name': video_name,
            'frame_number': frame_index
        }

        # Add each joint's (x, y, z) coordinate as a tuple in a separate column
        for joint_index in range(pose_data.shape[1]):
            row[f'joint_{joint_index + 1}'] = tuple(pose_data[frame_index, joint_index, :])
        
        row['label'] = label
        rows.append(row)

# Convert structured data into DataFrame for Excel
df_train = pd.DataFrame(rows)
df_train.to_excel(C.save_dir + "/test_aug_formatted.xlsx", index=False)

print("Data saved as both 'test_aug_formatted.xlsx' and 'test_aug_formatted.pkl' in the specified format.")

